In [1]:
#setup awal
from google.colab import drive, userdata
drive.mount('/content/drive')

import os
os.chdir('/content/drive/MyDrive/flood-hazard-bandung-bogor')

!git config user.email "latifalats.la@gmail.com"
!git config user.name "La01234"
!git pull origin main

print("✅ Setup selesai")

Mounted at /content/drive
remote: Enumerating objects: 69, done.
remote: Counting objects: 100% (69/69), done.
remote: Compressing objects: 100% (38/38), done.
remote: Total 58 (delta 33), reused 45 (delta 20), pack-reused 0 (from 0)
Unpacking objects: 100% (58/58), 6.49 MiB | 1.27 MiB/s, done.
From https://github.com/La01234/flood-hazard-bandung-bogor
 * branch            main       -> FETCH_HEAD
   6ab3664..b90f668  main       -> origin/main
Updating 6ab3664..b90f668
Fast-forward
 data/processed/features_bogor.parquet     | Bin 10134 -> 4810567 bytes
 models/xgb_bogor.json                     |   1 +
 notebooks/00_data_acquisition_bogor.ipynb |   2 +-
 notebooks/B_bogor_xgb.ipynb               |   2 +-
 outputs/distribusi_fitur_bogor.png        | Bin 114256 -> 190876 bytes
 outputs/evaluasi_xgb_bogor.png            | Bin 0 -> 115690 bytes
 outputs/korelasi_bogor.png                | Bin 175314 -> 170738 bytes
 7 files changed, 3 insertions(+), 2 deletions(-)
 create mode 100644 model

In [2]:
#install library
!pip install earthengine-api geemap geopandas rasterio scikit-learn xgboost \
             matplotlib seaborn folium streamlit optuna shap pyproj -q

print("✅ Library siap")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 49.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 425.6/425.6 kB 32.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 72.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 93.8 MB/s eta 0:00:00
✅ Library siap


In [4]:
#Autentikasi GEE
import ee
ee.Authenticate()
ee.Initialize(project='urban-analytic-492803')

dem_test = ee.Image('USGS/SRTMGL1_003')
print("GEE OK:", dem_test.getInfo()['id'])

GEE OK: USGS/SRTMGL1_003


In [5]:
#Import library dan load ROI
import ee
import geemap
import geopandas as gpd
import matplotlib.pyplot as plt
import os

# Load shapefile
bandung = gpd.read_file('data/boundaries/bandung_utm.gpkg')
bandung_geo = bandung.to_crs('EPSG:4326')

# Konversi ke GEE geometry
roi_bandung = geemap.geopandas_to_ee(bandung_geo).geometry()

MY_ROI  = roi_bandung
MY_CITY = 'bandung'

print("✅ ROI siap")
print("Bandung bounds:", bandung_geo.total_bounds)

✅ ROI siap
Bandung bounds: [107.54413543  -6.96973548 107.73951095  -6.83728677]


In [6]:
#Permanent water mask (dengan fix unmask)
jrc = ee.Image('JRC/GSW1_4/GlobalSurfaceWater')

# Fix: unmask dulu supaya Not() bekerja dengan benar
permanent_water = jrc.select('seasonality').gte(10) \
                     .unmask(0).toInt().rename('permanent_water')

water_stats = permanent_water.reduceRegion(
    reducer=ee.Reducer.sum(),
    geometry=MY_ROI,
    scale=30,
    maxPixels=1e10
)
print("Piksel air permanen:", water_stats.getInfo())

Piksel air permanen: {'permanent_water': 24}


In [7]:
#Built-up mask (dengan fix reproject)
# Fix: reproject ke 30m dan toInt supaya And() bekerja dengan benar
ghsl = ee.ImageCollection('JRC/GHSL/P2023A/GHS_BUILT_S') \
         .filter(ee.Filter.date('2020-01-01', '2021-01-01')) \
         .mosaic() \
         .select('built_surface') \
         .reproject(crs='EPSG:32748', scale=30)

built_up = ghsl.gt(0).toInt().rename('built_up')

# Study mask = terbangun DAN bukan air permanen
study_mask = built_up.And(permanent_water.Not()).rename('study_mask')

mask_stats = study_mask.reduceRegion(
    reducer=ee.Reducer.sum(),
    geometry=MY_ROI,
    scale=30,
    maxPixels=1e10
)
print("Piksel study area:", mask_stats.getInfo())

Piksel study area: {'study_mask': 179021.94901960774}


In [8]:
#DEM dan fitur topografi
dem    = ee.Image('USGS/SRTMGL1_003').rename('elevation')
slope  = ee.Terrain.slope(dem).rename('slope')
aspect = ee.Terrain.aspect(dem).rename('aspect')

slope_rad = slope.multiply(ee.Image(3.14159265).divide(180))
tan_slope  = slope_rad.tan().max(ee.Image(0.001))

flow_acc = ee.Image('WWF/HydroSHEDS/15ACC').rename('flow_acc')
twi      = flow_acc.divide(tan_slope).log().rename('TWI')

print("✅ Fitur topografi siap: elevation, slope, aspect, TWI")

✅ Fitur topografi siap: elevation, slope, aspect, TWI


In [9]:
#Sentinel-2 fitur spektral
s2 = ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED') \
       .filterBounds(MY_ROI) \
       .filterDate('2023-01-01', '2024-12-31') \
       .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 20)) \
       .median()

ndvi  = s2.normalizedDifference(['B8', 'B4']).rename('NDVI')
mndwi = s2.normalizedDifference(['B3', 'B11']).rename('MNDWI')
ndbi  = s2.normalizedDifference(['B11', 'B8']).rename('NDBI')

print("✅ Fitur spektral siap: NDVI, MNDWI, NDBI")

✅ Fitur spektral siap: NDVI, MNDWI, NDBI


In [10]:
#Sentinel-1 SAR
# Baseline: rata-rata 2022 (kondisi normal)
s1_baseline = ee.ImageCollection('COPERNICUS/S1_GRD') \
                .filterBounds(MY_ROI) \
                .filterDate('2022-01-01', '2022-12-31') \
                .filter(ee.Filter.eq('instrumentMode', 'IW')) \
                .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VV')) \
                .select('VV').mean().rename('SAR_VV_baseline')

# Flood event: banjir Bandung Maret 2024
s1_flood = ee.ImageCollection('COPERNICUS/S1_GRD') \
             .filterBounds(MY_ROI) \
             .filterDate('2024-03-01', '2024-03-31') \
             .filter(ee.Filter.eq('instrumentMode', 'IW')) \
             .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VV')) \
             .select('VV').mean().rename('SAR_VV_flood')

sar_change = s1_flood.subtract(s1_baseline).rename('SAR_change')

# Cek statistik SAR change
sar_stats = sar_change.reduceRegion(
    reducer=ee.Reducer.minMax(),
    geometry=MY_ROI,
    scale=30,
    maxPixels=1e10
)
print("SAR change stats:", sar_stats.getInfo())

SAR change stats: {'SAR_change_max': 18.443240903366213, 'SAR_change_min': -14.419289739763888}


In [11]:
#Cek threshold label banjir
for threshold in [-0.5, -1, -2, -3, -5]:
    flood_test = sar_change.lt(threshold) \
                           .And(permanent_water.Not()) \
                           .And(built_up) \
                           .rename('flood_label')
    stats = flood_test.reduceRegion(
        reducer=ee.Reducer.sum(),
        geometry=MY_ROI,
        scale=30,
        maxPixels=1e10
    )
    print(f"Threshold {threshold:5.1f} dB : {stats.getInfo()['flood_label']:,.0f} piksel")

Threshold  -0.5 dB : 47,762 piksel
Threshold  -1.0 dB : 24,241 piksel
Threshold  -2.0 dB : 4,692 piksel
Threshold  -3.0 dB : 1,066 piksel
Threshold  -5.0 dB : 178 piksel


In [12]:
# Cek total study area Bandung
mask_stats = study_mask.reduceRegion(
    reducer=ee.Reducer.sum(),
    geometry=MY_ROI,
    scale=30,
    maxPixels=1e10
)
total_study = mask_stats.getInfo()['study_mask']
print(f"Total study area: {total_study:,.0f} piksel")

for threshold, flood_px in [(-0.5, 47762), (-1, 24241), (-2, 4692), (-3, 1066), (-5, 178)]:
    pct = flood_px / total_study * 100
    print(f"Threshold {threshold:5.1f} dB : {flood_px:,} piksel ({pct:.1f}%)")

Total study area: 179,022 piksel
Threshold  -0.5 dB : 47,762 piksel (26.7%)
Threshold  -1.0 dB : 24,241 piksel (13.5%)
Threshold  -2.0 dB : 4,692 piksel (2.6%)
Threshold  -3.0 dB : 1,066 piksel (0.6%)
Threshold  -5.0 dB : 178 piksel (0.1%)


In [13]:
#Label banjir
THRESHOLD = -1  # 13.5% flood pixels — sesuai untuk Bandung

flood_label = sar_change.lt(THRESHOLD) \
                        .And(permanent_water.Not()) \
                        .And(built_up) \
                        .rename('flood_label')

flood_stats = flood_label.reduceRegion(
    reducer=ee.Reducer.sum(),
    geometry=MY_ROI,
    scale=30,
    maxPixels=1e10
)
print(f"Piksel banjir (threshold {THRESHOLD} dB):", flood_stats.getInfo())
print(f"Persentase: {flood_stats.getInfo()['flood_label']/179022*100:.1f}%")

Piksel banjir (threshold -1 dB): {'flood_label': 24241.10588235295}
Persentase: 13.5%


In [14]:
#Distance to river
rivers = ee.FeatureCollection('WWF/HydroSHEDS/v1/FreeFlowingRivers') \
           .filterBounds(MY_ROI)

river_raster = rivers.reduceToImage(
    properties=['RIV_ORD'],
    reducer=ee.Reducer.min()
).unmask(0).gt(0)

dist_river = river_raster.fastDistanceTransform().sqrt() \
                         .multiply(30).rename('dist_river')

print("✅ Distance to river siap")

✅ Distance to river siap


In [15]:
#Stack semua fitur
feature_stack = ee.Image.cat([
    dem,
    slope,
    aspect,
    twi,
    ndvi,
    mndwi,
    ndbi,
    s1_baseline,
    sar_change,
    dist_river,
    permanent_water,
    built_up,
    study_mask,
    flood_label
]).clip(MY_ROI).toFloat()

print("Band dalam feature stack:")
print(feature_stack.bandNames().getInfo())

Band dalam feature stack:
['elevation', 'slope', 'aspect', 'TWI', 'NDVI', 'MNDWI', 'NDBI', 'SAR_VV_baseline', 'SAR_change', 'dist_river', 'permanent_water', 'built_up', 'study_mask', 'flood_label']


In [16]:
#Export ke Google Drive
task = ee.batch.Export.image.toDrive(
    image=feature_stack,
    description=f'flood_features_{MY_CITY}',
    folder='FloodProject',
    fileNamePrefix=f'flood_features_{MY_CITY}',
    region=MY_ROI,
    scale=30,
    crs='EPSG:32748',
    maxPixels=1e10,
    fileFormat='GeoTIFF'
)

task.start()
print(f"✅ Export dimulai untuk {MY_CITY.upper()}")
print("Monitor di: https://code.earthengine.google.com/tasks")

✅ Export dimulai untuk BANDUNG
Monitor di: https://code.earthengine.google.com/tasks


In [18]:
#Pindah GeoTIFF ke repo
import shutil

# Cek semua file tif di FloodProject dulu
import datetime
folder = '/content/drive/MyDrive/FloodProject/'
for f in sorted(os.listdir(folder)):
    if 'bandung' in f:
        size = os.path.getsize(folder + f)/1e6
        mtime = os.path.getmtime(folder + f)
        print(f"{f} — {size:.1f} MB — {datetime.datetime.fromtimestamp(mtime)}")

flood_features_bandung (1).tif — 8.4 MB — 2026-06-07 16:47:39
flood_features_bandung.tif — 8.4 MB — 2026-06-06 09:07:44


In [ ]:
# Ganti nama file sesuai hasil cell di atas (pilih yang terbaru)
src = '/content/drive/MyDrive/FloodProject/flood_features_bandung.tif'  # sesuaikan
dst = 'data/raw/flood_features_bandung.tif'

shutil.copy(src, dst)
print(f"✅ File dipindah ke {dst}")
print(f"   Ukuran: {os.path.getsize(dst)/1e6:.1f} MB")